In [1]:
from backend.preprocess import Preprocessor

In [2]:
pre = Preprocessor()
train, val, report = pre.split(return_report=True)

for key, value in report.items():
    print(f"{key}: {value}")

Removed 0 duplicate rows by img_IDS.
[WARNING] 1 rows have no matching image files — dropped.
[MISLABEL] No conflicting labels found.
[DEDUP] Removed 0 pixel-identical duplicates.
rows_raw: 6249
removed_duplicate_ids: 0
rows_with_files: 6248
label_distribution_raw: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'Love': 694, 'You': 694, 'Friend': 694}
rows_missing_files: 1
removed_pixel_duplicates: 0
potential_mislabeled_groups: []
label_distribution_after_dedup: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'You': 694, 'Friend': 694, 'Love': 693}


Для текущей задачи распознавания жестов оптимально начать с базового пайплайна аугментаций: `Resize` + `Normalize`.  
Далее стоит добавить мягкие геометрические преобразования (`ShiftScaleRotate` с небольшими пределами), чтобы повысить устойчивость модели к сдвигам и изменению масштаба.  
Также полезны слабые фотометрические аугментации (`RandomBrightnessContrast`, `GaussianNoise`) с низкой вероятностью применения.

Агрессивные искажения (сильный blur, большие вырезания, чрезмерные деформации) лучше не использовать на старте, так как они могут ухудшить распознавание семантики жестов.  
`HorizontalFlip` нужно применять с осторожностью: для языка жестов отражение может менять смысл знака и потенциально вносить шум в разметку.  

Если в данных заметен дисбаланс классов, рекомендуется дополнительно использовать `class-balanced sampling` и более активные аугментации именно для минорных классов.